# ✨ Notebook 4: Content Structure Scoring

Uses LLMs to analyze presentation content structure:
- **Organization** (0-1): Logical flow, clear sections
- **Clarity** (0-1): Explanations, definitions, examples
- **Completeness** (0-1): Covers required elements
- **Engagement** (0-1): Hooks, transitions, audience awareness
- **Scientific Rigor** (0-1): Hypothesis, method, results, conclusion

**Approach:** Few-shot prompting with Gemini/Claude API
**Rubric:** Science fair presentation rubric (customizable)
**Output:** Structured JSON scores + qualitative feedback

In [ ]:
# Install dependencies
!pip install -q google-generativeai==0.7.0 anthropic==0.25.0 tenacity==8.2.3

import os
import json
import time
from pathlib import Path
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict
from tenacity import retry, stop_after_attempt, wait_exponential

print("✅ Dependencies installed")

In [ ]:
# Set up API keys (add your keys in Colab secrets or environment)
# In Colab: Use the key icon on the left sidebar to add secrets

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

if GEMINI_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini configured")
else:
    print("⚠️ GEMINI_API_KEY not set - add to Colab secrets")

if ANTHROPIC_API_KEY:
    import anthropic
    anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    print("✅ Anthropic configured")
else:
    print("⚠️ ANTHROPIC_API_KEY not set - add to Colab secrets")

In [ ]:
# Science Fair Presentation Rubric
RUBRIC = {
    "organization": {
        "weight": 0.20,
        "description": "Logical structure: intro, hypothesis, methods, results, conclusion",
        "levels": {
            1: "Disorganized, no clear sections",
            2: "Some structure but missing key sections",
            3: "Clear sections, minor flow issues",
            4: "Excellent logical flow, all sections present",
            5: "Masterful organization, seamless transitions"
        }
    },
    "clarity": {
        "weight": 0.20,
        "description": "Clear explanations, defines terms, uses examples/analogies",
        "levels": {
            1: "Unclear, jargon without explanation",
            2: "Partially clear, some terms undefined",
            3: "Mostly clear, good explanations",
            4: "Very clear, excellent use of examples",
            5: "Exceptionally clear, perfect for audience"
        }
    },
    "completeness": {
        "weight": 0.15,
        "description": "Covers all required elements for science fair",
        "levels": {
            1: "Missing most required elements",
            2: "Missing several key elements",
            3: "Covers most elements, minor gaps",
            4: "Covers all elements thoroughly",
            5: "Exceeds requirements, bonus insights"
        }
    },
    "engagement": {
        "weight": 0.15,
        "description": "Hooks audience, maintains interest, aware of audience level",
        "levels": {
            1: "Monotone, no engagement",
            2: "Minimal engagement attempts",
            3: "Good engagement, some hooks",
            4: "Strong engagement throughout",
            5: "Captivating, masterful audience connection"
        }
    },
    "scientific_rigor": {
        "weight": 0.30,
        "description": "Clear hypothesis, sound methodology, valid conclusions, data-driven",
    "levels": {
        1: "No hypothesis, flawed method, unsupported claims",
        2: "Weak hypothesis, method gaps, overstated conclusions",
        3: "Solid hypothesis, adequate method, reasonable conclusions",
        4: "Strong hypothesis, rigorous method, well-supported conclusions",
        5: "Publication-quality rigor, novel contribution"
    }
    }
}

# Required science fair elements
REQUIRED_ELEMENTS = [
    "title_and_intro",
    "problem_statement_or_question",
    "hypothesis",
    "background_research",
    "materials_and_methods",
    "data_and_results",
    "analysis_and_discussion",
    "conclusion",
    "future_work_or_applications"
]

print("Rubric loaded")
print(json.dumps(RUBRIC, indent=2))

In [ ]:
@dataclass
class ContentScore:
    organization: float
    clarity: float
    completeness: float
    engagement: float
    scientific_rigor: float
    overall: float
    feedback: str
    missing_elements: List[str]
    strengths: List[str]
    improvements: List[str]

    def to_dict(self):
        return asdict(self)

In [ ]:
# Build the prompt
def build_scoring_prompt(transcript: str) -> str:
    rubric_text = ""
    for key, val in RUBRIC.items():
        rubric_text += f"\n### {key.upper()} (weight: {val['weight']})\n"
        rubric_text += f"{val['description']}\n"
        for level, desc in val['levels'].items():
            rubric_text += f"  {level}: {desc}\n"
    
    elements_text = "\n".join([f"- {e}" for e in REQUIRED_ELEMENTS])
    
    prompt = f"""You are an expert science fair judge. Analyze this presentation transcript and score it.

TRANSCRIPT:
{transcript}

RUBRIC:
{rubric_text}

REQUIRED ELEMENTS (check completeness):
{elements_text}

OUTPUT FORMAT (JSON only):
{{
  "organization": <1-5>,
  "clarity": <1-5>,
  "completeness": <1-5>,
  "engagement": <1-5>,
  "scientific_rigor": <1-5>,
  "feedback": "<2-3 sentence overall assessment>",
  "missing_elements": ["<element1>", ...],
  "strengths": ["<strength1>", ...],
  "improvements": ["<improvement1>", ...]
}}

Score each dimension 1-5. Be critical but fair. Output ONLY valid JSON."""
    return prompt

print("Prompt builder ready")

In [ ]:
# Gemini scoring function
@retry(wait=wait_exponential(multiplier=1, min=2, max=10), stop=stop_after_attempt(3))
def score_with_gemini(transcript: str, model_name: str = "gemini-1.5-flash") -> ContentScore:
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY not set")
    
    model = genai.GenerativeModel(model_name)
    prompt = build_scoring_prompt(transcript)
    
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0.1,
            max_output_tokens=1024,
            response_mime_type="application/json"
        )
    )
    
    result = json.loads(response.text)
    
    # Convert 1-5 to 0-1
    for key in ['organization', 'clarity', 'completeness', 'engagement', 'scientific_rigor']:
        result[key] = (result[key] - 1) / 4
    
    # Weighted overall
    weights = {k: RUBRIC[k]['weight'] for k in ['organization', 'clarity', 'completeness', 'engagement', 'scientific_rigor']}
    result['overall'] = sum(result[k] * weights[k] for k in weights)
    
    return ContentScore(**result)

In [ ]:
# Anthropic/Claude scoring function
@retry(wait=wait_exponential(multiplier=1, min=2, max=10), stop=stop_after_attempt(3))
def score_with_claude(transcript: str, model_name: str = "claude-3-haiku-20240307") -> ContentScore:
    if not ANTHROPIC_API_KEY:
        raise ValueError("ANTHROPIC_API_KEY not set")
    
    prompt = build_scoring_prompt(transcript)
    
    response = anthropic_client.messages.create(
        model=model_name,
        max_tokens=1024,
        temperature=0.1,
        messages=[{"role": "user", "content": prompt}]
    )
    
    result = json.loads(response.content[0].text)
    
    for key in ['organization', 'clarity', 'completeness', 'engagement', 'scientific_rigor']:
        result[key] = (result[key] - 1) / 4
    
    weights = {k: RUBRIC[k]['weight'] for k in ['organization', 'clarity', 'completeness', 'engagement', 'scientific_rigor']}
    result['overall'] = sum(result[k] * weights[k] for k in weights)
    
    return ContentScore(**result)

In [ ]:
# Test with example transcript
EXAMPLE_TRANSCRIPT = """
Hello everyone, my name is Alex and today I'm presenting my project on the effects of 
different wavelengths of light on plant growth. My hypothesis was that blue light would 
promote the most growth because chlorophyll absorbs blue light most efficiently.
First, let me give you some background. Photosynthesis is the process plants use to 
convert light energy into chemical energy. Chlorophyll is the main pigment that absorbs 
light, and it has absorption peaks in the blue and red regions of the spectrum.
For my experiment, I grew 30 tomato plants under three conditions: blue LED, red LED, 
and white LED as a control. Each group had 10 plants. I measured height every 3 days 
for 4 weeks. All plants received the same water, soil, and temperature.
The results showed that the blue light group grew an average of 24 centimeters, 
the red light group grew 19 centimeters, and the white light control grew 21 centimeters. 
The difference between blue and red was statistically significant with p < 0.05.
This supports my hypothesis that blue light promotes more growth. However, I was 
surprised that the white light control outperformed red light. This might be because 
white light provides a full spectrum that supports other pigments too.
In conclusion, my hypothesis was supported. Blue light does promote the most growth 
in tomato plants. For future work, I'd like to test other wavelengths and different plant species.
Thank you for your time!
"""

# Test if API keys available
if GEMINI_API_KEY:
    print("Testing Gemini...")
    try:
        score = score_with_gemini(EXAMPLE_TRANSCRIPT)
        print(json.dumps(score.to_dict(), indent=2))
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping Gemini - no API key")

if ANTHROPIC_API_KEY:
    print("\nTesting Claude...")
    try:
        score = score_with_claude(EXAMPLE_TRANSCRIPT)
        print(json.dumps(score.to_dict(), indent=2))
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping Claude - no API key")

In [ ]:
# Batch scoring function
def score_transcripts_batch(transcripts: List[str], provider: str = "gemini") -> List[ContentScore]:
    """Score multiple transcripts with rate limiting."""
    scores = []
    
    for i, transcript in enumerate(transcripts):
        print(f"Scoring {i+1}/{len(transcripts)}...")
        try:
            if provider == "gemini":
                score = score_with_gemini(transcript)
            elif provider == "claude":
                score = score_with_claude(transcript)
            else:
                raise ValueError(f"Unknown provider: {provider}")
            scores.append(score)
        except Exception as e:
            print(f"Error on transcript {i}: {e}")
            scores.append(None)
        
        # Rate limiting
        time.sleep(1)
    
    return scores

# Save rubric and config
CONFIG_DIR = Path("/content/speech_analyzer/configs")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_DIR / "content_rubric.json", "w") as f:
    json.dump(RUBRIC, f, indent=2)

with open(CONFIG_DIR / "required_elements.json", "w") as f:
    json.dump(REQUIRED_ELEMENTS, f, indent=2)

print("Config saved")

## ✅ Content Scoring Module Complete!

**Files created:**
- `/content/speech_analyzer/configs/content_rubric.json`
- `/content/speech_analyzer/configs/required_elements.json`

**Usage:**
```python
from notebooks.content_scoring import score_with_gemini, score_with_claude

score = score_with_gemini(transcript)
print(f"Overall: {score.overall:.2f}")
print(f"Feedback: {score.feedback}")
```

**Next step:** Run `notebooks/05_integration_evaluation.ipynb` to combine all models.